In [1]:
import json
import os
import time
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path(__file__).resolve().parents[3] / ".env") if "__file__" in dir() else load_dotenv()

# ── Configuration ─────────────────────────────────────────────────────────────
API_KEY     = os.environ.get("DEEPSEEK_API_KEY")
MODEL       = "deepseek-v4-pro"
API_URL     = "https://api.deepseek.com/chat/completions"
PROMPTS_DIR = Path("../prompts_batches_20/batch_12")
CACHE_SPLIT_MARKER = "## SPECIES INFORMATION"

DELAY_BETWEEN_CALLS = 5
MAX_RETRIES         = 6
RETRY_DELAY         = 30
# ──────────────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError("DEEPSEEK_API_KEY environment variable is not set.")

prompt_files = sorted(PROMPTS_DIR.glob("*.json"))
print(f"Batch       : 12 / 20")
print(f"Model       : {MODEL}")
print(f"API URL     : {API_URL}")
print(f"Found {len(prompt_files)} prompt file(s)")

Batch       : 12 / 20
Model       : deepseek-v4-pro
API URL     : https://api.deepseek.com/chat/completions
Found 370 prompt file(s)


In [2]:
def split_prompt(prompt: str) -> tuple[str, str]:
    idx = prompt.find(CACHE_SPLIT_MARKER)
    if idx == -1:
        return prompt, ""
    return prompt[:idx].strip(), prompt[idx:].strip()


def call_deepseek(prompt: str) -> tuple[str, dict]:
    fixed_part, variable_part = split_prompt(prompt)

    if variable_part:
        messages = [
            {"role": "user",      "content": fixed_part},
            {"role": "assistant", "content": "Understood. Please provide the species information."},
            {"role": "user",      "content": variable_part},
        ]
    else:
        messages = [{"role": "user", "content": fixed_part}]

    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode())

    return data["choices"][0]["message"]["content"], data.get("usage", {})


# ── Main loop ──────────────────────────────────────────────────────────────────────
for idx, filepath in enumerate(prompt_files, 1):
    payload = json.loads(filepath.read_text())

    if payload.get("result") is not None:
        print(f"[SKIP]  {filepath.name}")
        continue

    print(f"[{idx:04d}/{len(prompt_files)}] {filepath.name} ...", end=" ", flush=True)

    fatal = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result, usage = call_deepseek(payload["prompt"])
            payload["result"] = result
            filepath.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
            cached = usage.get("prompt_cache_hit_tokens", 0)
            print(f"OK ({len(result)} chars | {usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out / {cached} cached)")
            break
        except urllib.error.HTTPError as e:
            body = e.read().decode(errors="replace")
            print(f"\n  HTTP {e.code} on attempt {attempt}: {body[:200]}")
            if e.code == 401:
                print("  Invalid API key — check DEEPSEEK_API_KEY. Aborting.")
                fatal = True
                break
            elif e.code == 404:
                print(f"  Model '{MODEL}' not found. Aborting.")
                fatal = True
                break
            elif e.code == 403:
                print("  Access forbidden — skipping this file.")
                break
            elif e.code == 429:
                wait = RETRY_DELAY * (2 ** (attempt - 1))
                print(f"  Rate limited — waiting {wait}s before retry...", flush=True)
                time.sleep(wait)
            elif attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
        except Exception as e:
            print(f"\n  Error on attempt {attempt}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    else:
        print(f"  FAILED after {MAX_RETRIES} attempts — skipping.")

    if fatal:
        break

    if idx < len(prompt_files):
        print(f"  Waiting {DELAY_BETWEEN_CALLS}s before next request...", flush=True)
        time.sleep(DELAY_BETWEEN_CALLS)

print("\nAll done.")

[0001/370] 4071_Microcondylaea_bonellii_abj.json ... OK (5956 chars | 19423 in / 1204 out / 0 cached)
  Waiting 5s before next request...
[0002/370] 4072_Microdipodops_megacephalus_stx.json ... OK (3702 chars | 20997 in / 792 out / 0 cached)
  Waiting 5s before next request...
[0003/370] 4073_Microdipodops_pallidus_stx.json ... OK (5448 chars | 21108 in / 1063 out / 0 cached)
  Waiting 5s before next request...
[0004/370] 4074_Microgadus_tomcod_abj.json ... OK (5654 chars | 17464 in / 1210 out / 0 cached)
  Waiting 5s before next request...
[0005/370] 4075_Microgale_cowani_stx.json ... OK (5400 chars | 20773 in / 1052 out / 0 cached)
  Waiting 5s before next request...
[0006/370] 4076_Microgale_dobsoni_stx.json ... OK (5887 chars | 20076 in / 1160 out / 0 cached)
  Waiting 5s before next request...
[0007/370] 4077_Microgale_drouhardi_stx.json ... OK (5142 chars | 21659 in / 1073 out / 0 cached)
  Waiting 5s before next request...
[0008/370] 4078_Microgale_mergulus_stx.json ... OK (6776

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────────────
done, failed = [], []
for filepath in sorted(PROMPTS_DIR.glob("*.json")):
    payload = json.loads(filepath.read_text())
    (done if payload.get("result") else failed).append(filepath.name)

print(f"Completed : {len(done)} / {len(done) + len(failed)}")
print(f"Missing   : {len(failed)}")
if failed:
    print("\nFiles still without result:")
    for f in failed:
        print(f"  {f}")